# TrustDTI Week 7 — Embedding Pilot

This notebook reproduces the **small encoder pilot** using 100 compound SMILES and 20 protein sequences. It saves portable float32 embeddings and validation summaries. It is not a DTI predictor.

## What this experiment can and cannot establish

It checks that pretrained encoders load, accept the sample inputs, return the expected shapes, produce finite values, and keep output rows aligned with input IDs. It does **not** establish DTI accuracy, calibration, or evidence-retrieval benefit.

In [ ]:
%pip -q install "transformers>=4.45,<5"

In [ ]:
import json, time, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import torch, transformers
from transformers import AutoModel, AutoTokenizer
from google.colab import files

print('Python runtime:', __import__('sys').version.split()[0])
print('PyTorch / Transformers:', torch.__version__, transformers.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

OUT = Path('/content/week07_outputs')
OUT.mkdir(parents=True, exist_ok=True)
compound_path = Path('/content/pilot_compounds.csv')
protein_path = Path('/content/pilot_proteins.csv')
if not (compound_path.exists() and protein_path.exists()):
    print('Upload pilot_compounds.csv and pilot_proteins.csv from TrustDTI_Week07_Pilot_Results/.')
    uploaded = files.upload()
    if not (compound_path.exists() and protein_path.exists()):
        raise FileNotFoundError('Both pilot CSVs must be uploaded to /content.')
compounds = pd.read_csv(compound_path)
proteins = pd.read_csv(protein_path)
assert len(compounds) == 100 and compounds['smiles'].notna().all()
assert len(proteins) == 20 and proteins['sequence'].notna().all()
assert proteins['accession'].is_unique
print('Pilot inputs verified:', len(compounds), 'compounds;', len(proteins), 'proteins.')

In [ ]:
# ChemBERTa compound embeddings
MODEL_ID = 'DeepChem/ChemBERTa-77M-MLM'
BATCH_SIZE = 16
assert torch.cuda.is_available(), 'Use a GPU runtime for the recorded pilot.'
device = torch.device('cuda')
smiles = compounds['smiles'].astype(str).tolist()
t0 = time.perf_counter()
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModel.from_pretrained(MODEL_ID).to(device).eval()
load_seconds = time.perf_counter() - t0
max_length = min(512, int(getattr(model.config, 'max_position_embeddings', 512)))
token_lengths = [len(tok.encode(s, add_special_tokens=True, truncation=False)) for s in smiles]
truncated = sum(n > max_length for n in token_lengths)
vectors = []
t0 = time.perf_counter()
with torch.inference_mode():
    for start in range(0, len(smiles), BATCH_SIZE):
        encoded = tok(smiles[start:start+BATCH_SIZE], padding=True, truncation=True, max_length=max_length, return_tensors='pt')
        ids = encoded['input_ids'].to(device)
        attn = encoded['attention_mask'].to(device)
        hidden = model(input_ids=ids, attention_mask=attn).last_hidden_state
        mask = attn.clone()
        mask[:, 0] = 0
        positions = torch.arange(mask.shape[1], device=device).expand_as(mask)
        last = positions.masked_fill(attn == 0, -1).max(dim=1).values
        mask[torch.arange(mask.shape[0], device=device), last] = 0
        mask = mask.unsqueeze(-1).to(hidden.dtype)
        vectors.append(((hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)).float().cpu().numpy())
        print(f'ChemBERTa embedded {min(start+BATCH_SIZE, len(smiles))}/{len(smiles)}')
chem_seconds = time.perf_counter() - t0
chem_vectors = np.concatenate(vectors, axis=0)
assert chem_vectors.shape == (100, 384) and np.isfinite(chem_vectors).all()
np.savez_compressed(OUT/'chemberta_pilot_embeddings.npz', embeddings=chem_vectors.astype(np.float32), inchikeys=compounds['inchikey'].astype(str).to_numpy(), smiles=compounds['smiles'].astype(str).to_numpy())
chem_summary = {'model_id':MODEL_ID,'device':torch.cuda.get_device_name(0),'num_compounds':len(compounds),'embedding_shape':list(chem_vectors.shape),'embedding_dimension':int(chem_vectors.shape[1]),'max_token_length':max(token_lengths),'truncated_compounds':int(truncated),'all_values_finite':bool(np.isfinite(chem_vectors).all()),'model_load_seconds':round(load_seconds,2),'inference_seconds':round(chem_seconds,2)}
(OUT/'chemberta_pilot_summary.json').write_text(json.dumps(chem_summary,indent=2))
print(json.dumps(chem_summary,indent=2))

In [ ]:
# ESM-2 protein embeddings
MODEL_ID = 'facebook/esm2_t6_8M_UR50D'
BATCH_SIZE = 4
sequences = proteins['sequence'].astype(str).str.upper().tolist()
t0 = time.perf_counter()
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModel.from_pretrained(MODEL_ID).to(device).eval()
load_seconds = time.perf_counter() - t0
max_length = min(1024, int(getattr(model.config, 'max_position_embeddings', 1026)) - 2)
token_lengths = [len(tok.encode(s, add_special_tokens=True, truncation=False)) for s in sequences]
truncated = sum(n > max_length for n in token_lengths)
vectors = []
t0 = time.perf_counter()
with torch.inference_mode():
    for start in range(0, len(sequences), BATCH_SIZE):
        encoded = tok(sequences[start:start+BATCH_SIZE], padding=True, truncation=True, max_length=max_length, return_tensors='pt')
        ids = encoded['input_ids'].to(device)
        attn = encoded['attention_mask'].to(device)
        hidden = model(input_ids=ids, attention_mask=attn).last_hidden_state
        mask = attn.clone()
        mask[:, 0] = 0
        positions = torch.arange(mask.shape[1], device=device).expand_as(mask)
        last = positions.masked_fill(attn == 0, -1).max(dim=1).values
        mask[torch.arange(mask.shape[0], device=device), last] = 0
        mask = mask.unsqueeze(-1).to(hidden.dtype)
        vectors.append(((hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)).float().cpu().numpy())
        print(f'ESM-2 embedded {min(start+BATCH_SIZE, len(sequences))}/{len(sequences)}')
esm_seconds = time.perf_counter() - t0
protein_vectors = np.concatenate(vectors, axis=0)
assert protein_vectors.shape == (20, 320) and np.isfinite(protein_vectors).all()
np.savez_compressed(OUT/'esm2_pilot_embeddings.npz', embeddings=protein_vectors.astype(np.float32), accessions=proteins['accession'].astype(str).to_numpy(), sequences=proteins['sequence'].astype(str).to_numpy())
esm_summary = {'model_id':MODEL_ID,'device':torch.cuda.get_device_name(0),'num_proteins':len(proteins),'embedding_shape':list(protein_vectors.shape),'embedding_dimension':int(protein_vectors.shape[1]),'max_sequence_length':max(map(len,sequences)),'truncated_proteins':int(truncated),'all_values_finite':bool(np.isfinite(protein_vectors).all()),'model_load_seconds':round(load_seconds,2),'inference_seconds':round(esm_seconds,2)}
(OUT/'esm2_pilot_summary.json').write_text(json.dumps(esm_summary,indent=2))
print(json.dumps(esm_summary,indent=2))

In [ ]:
# Final validation and download
with np.load(OUT/'chemberta_pilot_embeddings.npz', allow_pickle=False) as f:
    chem = {k:f[k] for k in f.files}
with np.load(OUT/'esm2_pilot_embeddings.npz', allow_pickle=False) as f:
    esm = {k:f[k] for k in f.files}
checks = {
 'ChemBERTa shape': chem['embeddings'].shape == (100,384),
 'ESM-2 shape': esm['embeddings'].shape == (20,320),
 'Compound InChIKeys align': np.array_equal(chem['inchikeys'].astype(str), compounds['inchikey'].astype(str).to_numpy()),
 'Compound SMILES align': np.array_equal(chem['smiles'].astype(str), compounds['smiles'].astype(str).to_numpy()),
 'Protein accessions align': np.array_equal(esm['accessions'].astype(str), proteins['accession'].astype(str).to_numpy()),
 'Protein sequences align': np.array_equal(esm['sequences'].astype(str), proteins['sequence'].astype(str).to_numpy()),
 'ChemBERTa values finite': bool(np.isfinite(chem['embeddings']).all()),
 'ESM-2 values finite': bool(np.isfinite(esm['embeddings']).all()),
}
for name, passed in checks.items():
    print(('PASS' if passed else 'FAIL') + ': ' + name)
if not all(checks.values()):
    raise RuntimeError('Pilot validation failed.')
report = {'project':'TrustDTI','course_week':7,'checks':checks,'chemberta':chem_summary,'esm2':esm_summary,'limitation':'Small encoder pilot only; not DTI predictive evaluation.'}
(OUT/'week07_pilot_validation.json').write_text(json.dumps(report,indent=2))
compounds.to_csv(OUT/'pilot_compounds.csv',index=False)
proteins.to_csv(OUT/'pilot_proteins.csv',index=False)
archive = Path('/content/TrustDTI_Week07_Pilot_Results.zip')
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(OUT.iterdir()):
        if path.is_file(): zf.write(path,arcname=path.name)
print('All checks passed. Results package:', archive)
files.download(str(archive))